# Lecture 13 — Loop Patterns: Flags, Best So Far, `break`, and `continue`
### EES 3350/5350 · Python in Earth Science · Friday, October 2, 2026

On Wednesday, we ended on two questions about the planet pairs:
1. Which pair of planets is the **most** alike in temperature?
2. If *no* pair were within 10 K, how would your code tell you?

Today answers both. Then we spend most of class on the hardest part of loops: **writing them yourself, from a blank cell.**

**Learning objectives for today**
- use a **flag** to tell "nothing matched" apart from "the loop is broken"
- keep track of the **best so far** (the smallest or largest value), along with the name that goes with it
- stop an inner loop early with `break`, and skip one pass with `continue`
- **plan a loop in comments** before writing any code

---
## Part 0 : set up

In [ ]:
planets = {
    "Mercury": {"solar_irradiance": 9082.7, "albedo": 0.068, "radius_km": 2439.7,
                "distance_mkm": 57.909,  "nasa_bb_temp_K": 439.6},
    "Venus":   {"solar_irradiance": 2601.3, "albedo": 0.77,  "radius_km": 6051.8,
                "distance_mkm": 108.210, "nasa_bb_temp_K": 226.6},
    "Earth":   {"solar_irradiance": 1361.0, "albedo": 0.294, "radius_km": 6371.0,
                "distance_mkm": 149.598, "nasa_bb_temp_K": 254.0},
    "Mars":    {"solar_irradiance": 586.2,  "albedo": 0.250, "radius_km": 3389.5,
                "distance_mkm": 227.956, "nasa_bb_temp_K": 209.8},
    "Jupiter": {"solar_irradiance": 50.26,  "albedo": 0.343, "radius_km": 69911,
                "distance_mkm": 778.479, "nasa_bb_temp_K": 109.9},
}

def blackbody_temp(S, a, stef_boltz=5.67e-8):
    """Predict a planet's blackbody surface temperature (K) from its solar constant (S) and albedo (a)."""
    T = ((1 - a) * S / (4 * stef_boltz)) ** (1/4)
    return T

### What does the first loop below do?
Take a couple seconds to look at the loop below...

In [ ]:
names = []
irradiances = [] 
temps = []
distances = []

for planet, props in planets.items():
    names.append(planet)
    irradiances.append(props["solar_irradiance"])
    temps.append(blackbody_temp(props["solar_irradiance"], props["albedo"]))
    distances.append(props["distance_mkm"])

for name, S, T, d in zip(names, irradiances, temps, distances):
    print(f"{name:>7}: S = {S:7.1f} W/m^2 | T = {T:5.1f} K | {d:5.1f} million km from the Sun")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>
It pulls the numbers we need out of the dictionary and into parallel lists. Sometimes it's hard to work with nested dictionaries, so let's work today with lists instead: one list per quanitity.

The lists then line up **by position**: position `i` in every list describes the same planet.
|`i`|`names[i]`|`irradiances[i]`|`temps[i]`|`distances[i]`|
|--|--|--|--|--|
|0|Mercury|9082.7| 439.5|  57.9|
|1|Venus|2601.3| 226.6 | 108.2 |
|2|Earth|1361.0| 255.1 | 149.6 |
|3|Mars|586.2 | 209.8 | 228.0 |
|4|Jupiter|50.3  | 109.8 | 778.5|

So `temps[i]` and `temps[j]`are the temperatures of planets `names[i]` and `names[j]`. Tht's why today's pair loops run over **positions** with `range(len(names))`: one `i` (or `j`) gets you into every list at once.

</details>

We also need Wednesday's albedo sweep:

In [ ]:
albedos = []

for i in range(10):
    albedos.append(i / 10)

print(albedos)

---
## Part 1 : when the loop says nothing (a flag)
Wednesday's Exercise 5 found pairs within **50 K**. Let's tighten that to **10 K**:

In [ ]:
for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = abs(temps[i] - temps[j])
        if diff <= 10:
            print(f"{names[i]} and {names[j]}: {diff:.1f} K apart")

<div class="alert alert-warning">

**Question:** That cell printed nothing. Did it work? List at least 2 *different* reasons a loop like this could print nothing.
    
</div>

> your answer here

The fix is a **flag**: a variable that starts as `False` ("haven't found one yet") and flips to `True` the moment a match happens. After the loops finish, we check it.

In [ ]:
threshold_k = 10

found = False                      # before BOTH loops: nothing found yet

for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = abs(temps[i] - temps[j])
        if diff <= threshold_k:
            print(f"{names[i]} and {names[j]}: {diff:.1f} K apart")
            found = True            # inside the IF we found one so make it TRUE

if not found:
    print(f"No pair of planets is within {threshold_k} K.")

Now the silence has a voice. Same code with a different threshold

In [ ]:
threshold_k = 20

found = False                      # before BOTH loops: nothing found yet

for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = abs(temps[i] - temps[j])
        if diff <= threshold_k:
            print(f"{names[i]} and {names[j]}: {diff:.1f} K apart")
            found = True            # inside the IF we found one so make it TRUE

if not found:
    print(f"No pair of planets is within {threshold_k} K.")

### What just happened?

|**piece**|**where it goes**|**why**|
|--|--|--|
|`found = False`| before **both** loops| there's one search, so there's one flag. It's created once.|
|`found = True`| inside the `if`| it flips the first time a pair matches, and nothing ever flips it back|
|`if not found:`| after **both** loops | you can only say "nothing matched" once **every** pair has been checked|

This is the same "where does it go?" rule as Lecture 11's counters. The only difference is that a flag holds `True`/`False` instead of a number.

<div class="alert alert-success">

**Tip**: `not found` is `True` when `found` is `False`. It reads like English and it means exactly the same thing as `found == False`.
    
</div>

<div class="alert alert-info">

### Exercise 1 : are any two orbits close together?
If we treat the orbits as circles (ignoring their slightly elliptical nature), the closest two planets can ever get is roughly the **gap between their distances** from the Sun. Fill in the three blanks so that the cell pritns every pair of orbits within `gap_limit`, **or** a message saying there are none.

Run it with `gap_likmit = 40`, then change it to `45` and run it again.
    
</div>

In [ ]:
# your answer here
gap_limit = 40   # million km

found = _______


for i in range(len(names)):
    for j in range(i+1, len(names)):
        gap = abs(________ - __________)
        if gap <= gap_limit:
            print(f"{names[i]} and {names[j]}: orbits {gap:.1f} million km apart.")
            found = ______
if not found:
    print(f"No two orbits are within {gap_limit} million km.")

<div class="alert alert-info">

### Exercise 2: Debug me! (no error message)
This cell is supposed to find Venus and Mars (16.8 K apart) and **not** print the "no pair" message. Run it, The output contradicts itself.

Fix it by **moving one line**.
    
</div>

In [ ]:
# this cell runs, but something is wrong. Find and fix the bug
threshold_k = 20

for i in range(len(names)):
    for j in range(i+1, len(names)):
        found = False
        diff = abs(temps[i] - temps[j])
        if diff <= threshold_k:
            print(f"{names[i]} and {names[j]}: {diff:.1f} K apart")
            found = True

if not found:
    print(f"No pair of planets is within {threshold_k} K.")

---
## Part 2 : the best so far
Back to Wednesday's other question: which pair is the **most** alike?

On the Lab 4 worksheet you found the wettest season by first finding the `max()`, then looping *again* to find which season it belonged to. Today we can do both jobs in **one pass**, but keeping track of the best result we've seen *so far*:

In [ ]:
best_diff = 10000  # start BIGGER than any real difference
best_pair = ""

for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = abs(temps[i] - temps[j])
        if diff < best_diff:                         # tests if this pair is better than the best so far
            best_diff = diff                         # if it is, then we need to remember the number
            best_pair = f"{names[i]} and {names[j]}" # and then we make a name that goes with it

print(f"Most alike: {best_pair}, {best_diff:.1f} K apart")

#### What just happened?

Here is every pass, in order, `best_diff` only ever changes when a bair beats it:

| **pair** | `diff` (K) | `diff < best_diff`? | `best_diff` after this pass |
|:-|-:|:-:|-:|
| *(before the loops)* | | | 10000 |
| Mercury vs Venus | 212.9 | **yes** | 212.9 |
| Mercury vs Earth | 184.4 | **yes** | 184.4 |
| Mercury vs Mars | 229.7 | no | 184.4 |
| Mercury vs Jupiter | 329.7 | no | 184.4 |
| Venus vs Earth | 28.5 | **yes** | 28.5 |
| Venus vs Mars | 16.8 | **yes** | 16.8 |
| Venus vs Jupiter | 116.8 | no | 16.8 |
| Earth vs Mars | 45.3 | no | 16.8 |
| Earth vs Jupiter | 145.3 | no | 16.8 |
| Mars vs Jupiter | 100.0 | no | 16.8 |

The best value goes *8down** four times and then never changes again: no later pair beats Venus or Mars.

So there's three pieces, with the same placement rule as the flag:
|**place**|**where it goes**|
|:-|:-|
|`best_diff = 10000` and `best_pair = ""` | before **both** loops|
|the comparison and both updates | inside the inner loop, the updates inside the `if`|
|`print(...)`|after **both** loops|

> **Why 100000?** It has to be bigger than any real difference so the **first** pair always wins and becomes the first "*best*". Any big number works, The tidy versino is `float("inf")`, Python's infinity, which is bigger than every number.

<div class="alert alert-danger">

**Common mistake:** starting `best_diff = 0` when you're looking for the **smallest** value. No difference is ever smaller than 0, so the `if` is never ture, and you get `best_pair = ""`. 
- Looking for the smallest? start big.
- Looking for the largest? start small.
    
</div>

<div class="alert alert-info">

### Exercise 3: the most *different* pair

Now find the pair of planets whose temperatures are **farthest** apart. Think about the starting value, and which way the comparison has to point.
    
</div>

In [ ]:
# your answer here
best_diff = _____
best_pair = ""

for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = abs(temps[i] - temps[j])
        if diff ____ best_diff:
            best_diff = _____
            best_pair = _____

print(f"Most different: {best_pair}, {best_diff:.1f} K apart")

<div class="alert alert-info">

### Exercise 4: Debug me! (no error message)

This was supposed to find the most-alike pair. It reports **16.8 K**, which is right, but pairs that number with **Mars and Jupiter**, which are 100 K apart.

Fix it by **changing the indentation of one line**.
    
</div>

In [ ]:
# this cell runs, but something is wrong. Find and fix the bug.

best_diff = 10000
best_pair = ""

for i in range(len(names)):
    for j in range(i+1, len(names)):
        diff = abs(temps[i] - temps[j])
        if diff < best_diff:
            best_diff = diff
        best_pair = f"{names[i]} and {names[j]}"

print(f"Most alike: {best_pair}, {best_diff:.1f} K apart")

---
## Part 3 : `break` and `continue`

### `break`: stop the inner loop early

New question, and a different kind of nested loop. instead of pairs from **one** list, the outer loop walks the **planets** and the inner loop walks a **second** list, `albedos` (just like Wednesday's Part 1).

> For each planet, what is the **lowest** albedo in teh sweep that would put it below freezing (273.15 K)?

A higher albedo only makes a planet colder, so once we find the first one below freezing there's no reason to check the rest. `break` stops the inner loop right there:

In [ ]:
freezing_k = 273.15

for name, S in zip(names, irradiances):      # outer: one pass per planet
    for albedo in albedos:                   # inner: one pass per albedo
        T = blackbody_temp(S, albedo)
        if T < freezing_k:
            print(f"{name:>7}: below freezing once albedo reaches {albedo}")
            break                            # stop checking albedos for THIS planet

<div class="alert alert-warning">

**Question**: `break ran five times. Why did we still get a line for **every** planet? What exactly did **break** stop?
    
</div>

> your answer here

<div class="alert alert-success">

**Tip**: `break` always exists only the in the **innermost** loop that it's in. Breaking out of *both* loops at once is fidly, and we don't need it. When you want to stop a whole search, use a flag.
    
</div>

### `continue`: skip just this pass

`continue` doesn't stop anything. It skips the **rest of this pass** and jumps straight to the next value. Jupiter is a gas giant with no solid surface, so here's every unique pair of **rocky** planets:


In [ ]:
for i in range(len(names)):
    for j in range(i+1, len(names)):
        if names[i] == "Jupiter" or names[j] == "Jupiter":
            continue
        print(names[i], "vs", names[j])

This will produce six pairs instead of ten: the four Jupiter pairs got skipped.

||**what it does**| **in a nested loop**|
|--|--|--|
|`break`| stop this loop **completely**| the inner loop ends; the outer loop moves to its next value|
|`continue`|skip the **rest of this pass**| the inner loop jumps to its next value|

<div class="alert alert-info">

### Exercise 5 : big temperature gaps between rocky planets

Count the unique pairs of **rocky** planets whose temperatures are **more than 100K** apart. Use `continue` to skip any pair that includes Jupiter. Print the final count.
    
</div>

In [ ]:
# make a counter, starting at 0
# loop over positions i
    # loop over positions j, starting one after i
        # if either planet is Jupiter, skip this pair with continue
        # if the temperature difference is more than 100 K, add 1 to the counter
# print the count (after both loops)

---
## Part 4 : on your own

Everything from here on starts from a **blank cell**. When a blank cell feels hard, don't start with code. Start with a plan.

<div class="alert alert-success">

**Tip: plan in comments first**. Before writing any code, answer these four questions as `#` comments in your cell, then write the code underneath each comment:

1. **What am I looping over?** Unique pairs? `i` and `j` from `i+1`? Or every planet x every albedo?
2. **What do I need before the loops?** A counter? A flag? a best-so-far value and name?
3. **What happens on each pass?** What do I compute, what do I compare, and what do I update?
4. **What happens after the loops?** Print the result? Check the flag?

That's exactly what the comments-only exercises have been giving you. Now you write them.
    
</div>

<div class="alert alert-info">

### Exercise 6 : the most distant pair of orbits

Which pair of planets has the **biggest** gap between their distances from the Sun? Print the pair and the gap in millions of km.
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exercise 7 : what albedo would make each planet feel like Earth?

Earth's actual average surface temperature is **288 K**. For **each planet**, find the albedo in `albedos` whose blackbody temperature comes **closest** to 288 K. Print one line per planet with the albedo and how far off it is,

This loops over planets x albedos, like the `break` demo. *Each planet needs its own best. So where do the best-so-far variables have to be created this time?*
    
</div>

In [ ]:
# your answer here


<div class="alert alert-info">

### Exercise 8 : a hundred times the sunlight?

Does any planet get **more than 100 times** as much sunlight (check irradiance) as another planet? Print every pair that does, with the ratio. If none do, Print a message saying so.

Then change 100 to 200 and run it again. *Since `i` is always the planet closer to the Sun, `irradiances[i] / irradiances[j]` is always bigger than 1.
    
</div>

In [ ]:
# your answer here

## Parallel dictionaries
You can store the same data as **dictionaries keyed by planet name** instead of lists. You met `dict(zip(...))` in Lecture 10:

In [ ]:
temp_by_planet = dict(zip( names, temps))
distance_by_planet = dict(zip(names, distances))

print(temp_by_planet["Venus"], distance_by_planet["Venus"])

Same numbers, but now you look them up by **name** instead of by position. In a pairs loop, grab the two names first (`a = names[i]`, `b = names[j]`), the use them as keys: `temp_by_planet[a]`.

<div class="alert alert-info">

### Exercise 9 : should the farther planet be colder?

Use the two dictionaries above, not the lists.

You'd expect a planet farther form teh Sun to be colder. For every unique pair, `names[j]` is the planet farther from teh Sun. Pring any pair where the **farther** planet is actually **warmer** than the closer one. If there are none, print a message saying so.
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exercise 10 : all three patterns in one loop

Using **rocky planets only** (skip Jupiter with `continue`, write *one* nested loop that does two jobs at once:
1. finds the rocky pair whose orbits are **closest** together (best so far), and
2. uses a flag to check whether **any** rocky pair of orbits is within 30 million km.

After the loop, print the closest pair. Then, if no pair was within 30 million km, print a message saying so.
    
</div>

In [ ]:
# your answer here

---
## A useful table

|**pattern**|**before the loops**|**inside**|**after the loops**|
|--|--|--|--|
|flag | `found = False`| `found = True` inside the `if`| `if not found:` print a "none" message|
|best so far (smallest)| `best = 10000` (or `float("inf")`), `best_name - ""`| `if x < best:` update **both** | print `best_name` and `best`|
|best so far (largest)| `best = 0`, `best_name = ""`| `if x > best:` update **both** | print `best_name` and `best`|
|one result **per outer item**| -- | create the counter / flag / best **inside the outer loop, before the inner loop** | print **after the inner loop**, still inside the outer one
|`break`| -- | stop the **inner** loop early | the outer loop keeps going|
|`continue`| -- | skip the rest of **this pass** | --|
| blank cell?| write the plan as # comments first | -- | -- |